## Statistical Testing

In [ ]:
#Load cleaned data

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind

df = pd.read_csv(
    "../data/cleaned/healthcare_appointments_cleaned.csv"
)

alpha = 0.05

print(df.shape)

(110522, 15)


## Chi-Square Test

In [ ]:
#Create a Chi-Square function

def chi_square_test(data, column, target="No_show"):
    
    table = pd.crosstab(data[column], data[target])
    
    chi2, p, dof, expected = chi2_contingency(table)
    
    result = "Significant" if p < alpha else "Not Significant"
    
    print(f"\n--- {column} vs {target} ---")
    print("Chi-square statistic:", round(chi2, 4))
    print("p-value:", round(p, 6))
    print("Degrees of freedom:", dof)
    print("Result:", result)
    
    return chi2, p

## Test categorical variables

In [3]:
chi_square_test(df, "Gender")


--- Gender vs No_show ---
Chi-square statistic: 1.8623
p-value: 0.172358
Degrees of freedom: 1
Result: Not Significant


(np.float64(1.8623155959345414), np.float64(0.1723577850309428))

In [4]:
chi_square_test(df, "Scholarship")


--- Scholarship vs No_show ---
Chi-square statistic: 93.7812
p-value: 0.0
Degrees of freedom: 1
Result: Significant


(np.float64(93.78124536262366), np.float64(3.5239509691014426e-22))

In [5]:
chi_square_test(df, "Hypertension")


--- Hypertension vs No_show ---
Chi-square statistic: 140.3204
p-value: 0.0
Degrees of freedom: 1
Result: Significant


(np.float64(140.32041399921096), np.float64(2.2654118082422854e-32))

In [6]:
chi_square_test(df, "Diabetes")


--- Diabetes vs No_show ---
Chi-square statistic: 25.2424
p-value: 1e-06
Degrees of freedom: 1
Result: Significant


(np.float64(25.242403234224557), np.float64(5.055831574799051e-07))

In [7]:
chi_square_test(df, "Alcoholism")


--- Alcoholism vs No_show ---
Chi-square statistic: 0.0014
p-value: 0.969638
Degrees of freedom: 1
Result: Not Significant


(np.float64(0.0014487790158767197), np.float64(0.9696375873068587))

In [8]:
chi_square_test(df, "Handicap")


--- Handicap vs No_show ---
Chi-square statistic: 7.4867
p-value: 0.112297
Degrees of freedom: 4
Result: Not Significant


(np.float64(7.486703156959369), np.float64(0.11229705875174573))

In [9]:
chi_square_test(df, "SMS_received")


--- SMS_received vs No_show ---
Chi-square statistic: 1768.0701
p-value: 0.0
Degrees of freedom: 1
Result: Significant


(np.float64(1768.0700627519273), np.float64(0.0))

## Chi-square results table

In [10]:
categorical_variables = [
    "Gender",
    "Scholarship",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "Handicap",
    "SMS_received"
]

chi_results = []

for column in categorical_variables:
    
    table = pd.crosstab(df[column], df["No_show"])
    
    chi2, p, dof, expected = chi2_contingency(table)
    
    chi_results.append({
        "Variable": column,
        "Chi-square": chi2,
        "p-value": p,
        "Degrees of Freedom": dof,
        "Significant": "Yes" if p < alpha else "No"
    })

chi_results_df = pd.DataFrame(chi_results)

chi_results_df

,Variable,Chi-square,p-value,Degrees of Freedom,Significant
0,Gender,1.862316,1.723578e-01,1,No
1,Scholarship,93.781245,3.523951e-22,1,Yes
2,Hypertension,140.320414,2.265412e-32,1,Yes
3,Diabetes,25.242403,5.055832e-07,1,Yes
4,Alcoholism,0.001449,9.696376e-01,1,No
5,Handicap,7.486703,1.122971e-01,4,No
6,SMS_received,1768.070063,0.000000e+00,1,Yes


## T-Test: Age

question

Is there a significant difference in the mean age between patients who attended and patients who did not attend?

In [23]:
from scipy.stats import ttest_ind

attended_age = df.loc[
    df["No_show"] == "No",
    "Age"
]

noshow_age = df.loc[
    df["No_show"] == "Yes",
    "Age"
]

t_stat_age, p_value_age = ttest_ind(
    attended_age,
    noshow_age,
    equal_var=False
)

print("Age T-statistic:", t_stat_age)
print("Age p-value:", p_value_age)

if p_value_age < alpha:
    print("Reject H0: There is a significant difference in mean age.")
else:
    print("Fail to reject H0: There is no significant difference in mean age.")

Age T-statistic: 20.82872111346898
Age p-value: 8.704289224373237e-96
Reject H0: There is a significant difference in mean age.


## T-Test: WaitingDays

In [24]:
attended_waiting = df.loc[
    df["No_show"] == "No",
    "WaitingDays"
]

noshow_waiting = df.loc[
    df["No_show"] == "Yes",
    "WaitingDays"
]

t_stat_waiting, p_value_waiting = ttest_ind(
    attended_waiting,
    noshow_waiting,
    equal_var=False
)

print("WaitingDays T-statistic:", t_stat_waiting)
print("WaitingDays p-value:", p_value_waiting)

if p_value_waiting < alpha:
    print("Reject H0: There is a significant difference in mean waiting time.")
else:
    print("Fail to reject H0: There is no significant difference in mean waiting time.")

WaitingDays T-statistic: -58.28738009835706
WaitingDays p-value: 0.0
Reject H0: There is a significant difference in mean waiting time.


## Display the group means 

In [17]:
print("Mean age - Attended:", attended_age.mean())
print("Mean age - No-show:", noshow_age.mean())

Mean age - Attended: 37.79049519317976
Mean age - No-show: 34.31787218786412


In [22]:
print("Mean waiting days - Attended:", attended_waiting.mean())
print("Mean waiting days - No-show:", noshow_waiting.mean())

Mean waiting days - Attended: 8.754659441320515
Mean waiting days - No-show: 15.835484449224701


## Create the T-Test results table

In [18]:
ttest_results = pd.DataFrame({
    "Variable": ["Age", "WaitingDays"],
    "Attended Mean": [
        attended_age.mean(),
        attended_waiting.mean()
    ],
    "No-show Mean": [
        noshow_age.mean(),
        noshow_waiting.mean()
    ],
    "T-statistic": [
        t_stat_age,
        t_stat_waiting
    ],
    "p-value": [
        p_value_age,
        p_value_waiting
    ],
    "Significant": [
        "Yes" if p_value_age < alpha else "No",
        "Yes" if p_value_waiting < alpha else "No"
    ]
})

ttest_results

,Variable,Attended Mean,No-show Mean,T-statistic,p-value,Significant
0,Age,37.790495,34.317872,20.828721,8.704289e-96,Yes
1,WaitingDays,8.754659,15.835484,-58.287380,0.000000e+00,Yes


## Save the statistical results

In [19]:
chi_results_df.to_csv(
    "../data/cleaned/chi_square_results.csv",
    index=False
)

In [20]:
ttest_results.to_csv(
    "../data/cleaned/t_test_results.csv",
    index=False
)

## Final Testing Summary

In [21]:
print("STATISTICAL TESTING SUMMARY")
print("=" * 40)

print(f"Significance level (alpha): {alpha}")

print("\nChi-Square Tests:")
print(chi_results_df)

print("\nT-Tests:")
print(ttest_results)

STATISTICAL TESTING SUMMARY
Significance level (alpha): 0.05

Chi-Square Tests:
       Variable   Chi-square       p-value  Degrees of Freedom Significant
0        Gender     1.862316  1.723578e-01                   1          No
1   Scholarship    93.781245  3.523951e-22                   1         Yes
2  Hypertension   140.320414  2.265412e-32                   1         Yes
3      Diabetes    25.242403  5.055832e-07                   1         Yes
4    Alcoholism     0.001449  9.696376e-01                   1          No
5      Handicap     7.486703  1.122971e-01                   4          No
6  SMS_received  1768.070063  0.000000e+00                   1         Yes

T-Tests:
      Variable  Attended Mean  No-show Mean  T-statistic       p-value  \
0          Age      37.790495     34.317872    20.828721  8.704289e-96   
1  WaitingDays       8.754659     15.835484   -58.287380  0.000000e+00   

  Significant  
0         Yes  
1         Yes  


## Statistical Testing Conclusion

Statistical hypothesis testing was performed using a significance level of 0.05.

Chi-square tests were used to examine the association between categorical variables and appointment no-show status. Significant associations were found for Scholarship, Hypertension, Diabetes, and SMS_received. Gender, Alcoholism, and Handicap did not show statistically significant associations with appointment no-show status.

Independent t-tests were performed to compare the mean Age and WaitingDays between attended and no-show appointments. Both variables showed statistically significant differences.

The results indicate that several patient and appointment-related factors are statistically associated with appointment attendance. However, statistical association does not imply causation.